# Un lot trop grand : garder le même objectif en le découpant

Pondérer les pertes et accumuler les gradients avant une seule mise à jour.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#moyennes-et-microbatches).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Le dernier lot contient parfois moins d’exemples. Pour le bilan de l’époque, deux moyennes de lots ne suffisent pas : il faut aussi leurs effectifs.

| Groupe fictif | Moyenne | Effectif | Somme retrouvée |
| --- | --- | --- | --- |
| Premier | 2 | 8 | 16 |
| Dernier | 5 | 2 | 10 |

La moyenne par exemple vaut (16+10)/10=2,6. La moyenne des deux lots vaut 3,5 : elle donne le même poids aux groupes.

Cette règle sert aussi lorsque le lot ne tient pas en mémoire. Un microlot est un morceau traité séparément. Nous accumulons ses gradients en gardant les poids fixes jusqu’au dernier morceau.

Prenons un calcul indépendant de l’audio : x=[1,2,3,4], cible=2x et prédiction=poids×x avec poids=1. Les quatre erreurs carrées valent 1,4,9,16 ; leur moyenne vaut 7,5 et son gradient −15.

Chaque perte_locale est déjà une moyenne. Le groupe de trois contribue pour 3/4 ; le dernier pour 1/4. backward ajoute leurs contributions dans grad. Aucun step n’est placé dans la boucle.

Le gradient total retrouve −15, et le poids devient 2,5, comme avec le lot entier. Diviser chaque perte par le nombre de morceaux échouerait ici : leurs effectifs ne sont pas égaux.

Avec quatre microlots de même taille, chaque fraction vaudrait en revanche 1/4 : diviser chaque perte moyenne par quatre serait alors correct.

Cette équivalence suppose un calcul déterministe et une perte séparable entre exemples. BatchNorm, Dropout ou une perte entre exemples peuvent la modifier. Pour une moyenne par trame valide, le dénominateur serait le nombre de trames valides, pas le nombre de fichiers.

## Démonstration 1 · Comparer le lot entier à deux microlots

Contrôler le gradient avant de comparer les poids après le pas.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Les effectifs permettent de retrouver la moyenne par exemple.

In [4]:
moyenne_globale = (2. * 8 + 5. * 2) / (8 + 2)
moyenne_des_lots = (2. + 5.) / 2


In [5]:
observer('moyenne_globale', moyenne_globale)
observer('moyenne_des_lots', moyenne_des_lots)

moyenne_globale = 2.6
moyenne_des_lots = 3.5


### Choisir les paramètres et leur mise à jour

La MSE moyenne des quatre exemples produit le gradient de référence.

In [6]:
X = torch.tensor([1., 2., 3., 4.])
cibles = 2 * X
poids_complet = torch.tensor(1., requires_grad=True)
optim_complet = torch.optim.SGD([poids_complet], lr=0.1)


In [7]:
observer('X', X)
observer('cibles', cibles)
observer('poids_complet', poids_complet)
observer('optim_complet', optim_complet)

X forme (4,) type torch.float32 requires_grad False
  valeurs : [1.0, 2.0, 3.0, 4.0]
cibles forme (4,) type torch.float32 requires_grad False
  valeurs : [2.0, 4.0, 6.0, 8.0]
poids_complet forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
perte_complete = ((poids_complet * X - cibles) ** 2).mean()


In [9]:
observer('perte_complete', perte_complete)

perte_complete forme () type torch.float32 requires_grad True
  valeurs : 7.5


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
perte_complete.backward()


In [11]:
observer_entrainement({'moyenne_globale': moyenne_globale, 'moyenne_des_lots': moyenne_des_lots, 'X': X, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete}, montrer_poids=False)

poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
gradient_complet = poids_complet.grad.item()


In [13]:
observer('gradient_complet', gradient_complet)

gradient_complet = -15.0


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
optim_complet.step()


In [15]:
observer_entrainement({'moyenne_globale': moyenne_globale, 'moyenne_des_lots': moyenne_des_lots, 'X': X, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete, 'gradient_complet': gradient_complet}, montrer_poids=True)

poids_complet forme () type torch.float32 requires_grad True
  valeurs : 2.5
poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0


### Choisir les paramètres et leur mise à jour

Les gradients sont remis à zéro avant l’ensemble des morceaux.

In [16]:
poids_decoupe = torch.tensor(1., requires_grad=True)
optim_decoupe = torch.optim.SGD([poids_decoupe], lr=0.1)


In [17]:
observer('poids_decoupe', poids_decoupe)
observer('optim_decoupe', optim_decoupe)

poids_decoupe forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Remettre les gradients à zéro

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [18]:
optim_decoupe.zero_grad()


In [19]:
observer_entrainement({'moyenne_globale': moyenne_globale, 'moyenne_des_lots': moyenne_des_lots, 'X': X, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete, 'gradient_complet': gradient_complet, 'poids_decoupe': poids_decoupe, 'optim_decoupe': optim_decoupe}, montrer_poids=False)

poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0
poids_decoupe.grad = None


### Répéter la même étape sur les données

Les fractions d’effectif reconstruisent le même objectif global.

In [20]:
for debut, fin in [(0, 3), (3, 4)]:
    x = X[debut:fin]
    cible = cibles[debut:fin]
    perte_locale = ((poids_decoupe * x - cible) ** 2).mean()
    contribution = perte_locale * len(x) / len(X)
    contribution.backward()


### Calculer l’état suivant

Le poids ne change qu’après le second backward.

In [21]:
gradient_decoupe = poids_decoupe.grad.item()


In [22]:
observer('gradient_decoupe', gradient_decoupe)

gradient_decoupe = -15.0


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [23]:
optim_decoupe.step()


In [24]:
observer_entrainement({'moyenne_globale': moyenne_globale, 'moyenne_des_lots': moyenne_des_lots, 'X': X, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete, 'gradient_complet': gradient_complet, 'poids_decoupe': poids_decoupe, 'optim_decoupe': optim_decoupe, 'gradient_decoupe': gradient_decoupe}, montrer_poids=True)

poids_complet forme () type torch.float32 requires_grad True
  valeurs : 2.5
poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0
poids_decoupe forme () type torch.float32 requires_grad True
  valeurs : 2.5
poids_decoupe.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [25]:
print('moyenne des exemples :', moyenne_globale, 'des lots :', moyenne_des_lots)
print('gradients :', gradient_complet, gradient_decoupe)
print('poids :', poids_complet.item(), poids_decoupe.item())

moyenne des exemples : 2.6 des lots : 3.5
gradients : -15.0 -15.0
poids : 2.5 2.5


In [26]:
assert bool(moyenne_globale == 2.6 and moyenne_des_lots == 3.5), 'moyennes-et-microbatches/main: moyenne_globale == 2.6 and moyenne_des_lots == 3.5'
assert bool(perte_complete.item() == 7.5 and gradient_complet == -15), 'moyennes-et-microbatches/main: perte_complete.item() == 7.5 and gradient_complet == -15'
assert bool(gradient_decoupe == gradient_complet), 'moyennes-et-microbatches/main: gradient_decoupe == gradient_complet'
assert bool(poids_decoupe.item() == poids_complet.item() == 2.5), 'moyennes-et-microbatches/main: poids_decoupe.item() == poids_complet.item() == 2.5'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Le découpage économise les activations des morceaux, à condition de ne pas conserver leurs graphes après backward.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un lot de huit exemples a une perte moyenne de 1 ; le dernier de deux exemples, de 3. Quelle moyenne globale ?

In [27]:
exo_tailles = [8,2]
exo_pertes = [1.,3.]
exo_moyenne = ...  # Remplacer les points par votre réponse.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-sum(n*p for n,p in zip(exo_tailles,exo_pertes))/sum(exo_tailles)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Additionnez d’abord les contributions 8×1 et 2×3, puis divisez par dix exemples.

</details>

### Exercice 2

Le lot x=[1,2,3,4], cible=2x, est coupé en trois exemples puis un. Donnez les coefficients à appliquer aux deux MSE moyennes avant backward.

In [28]:
exo_x = torch.tensor([1.,2.,3.,4.])
exo_cibles = 2*exo_x
exo_p = torch.tensor(1.,requires_grad=True)
exo_coefficients = ...  # Remplacer les points par votre réponse.
if exo_coefficients is Ellipsis or exo_coefficients is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    for exo_coefficient,(exo_a,exo_b) in zip(exo_coefficients,[(0,3),(3,4)]):
        exo_locale=((exo_p*exo_x[exo_a:exo_b]-exo_cibles[exo_a:exo_b])**2).mean()
        (exo_coefficient*exo_locale).backward()
    assert abs(exo_p.grad.item()+15) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Chaque fraction représente un effectif. Gardez les poids fixes et ne remettez pas les gradients à zéro entre les morceaux.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_tailles = [8,2]
exo_pertes = [1.,3.]
exo_moyenne = 1.4
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-sum(n*p for n,p in zip(exo_tailles,exo_pertes))/sum(exo_tailles)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([1.,2.,3.,4.])
exo_cibles = 2*exo_x
exo_p = torch.tensor(1.,requires_grad=True)
exo_coefficients = (3/4,1/4)
if exo_coefficients is Ellipsis or exo_coefficients is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    for exo_coefficient,(exo_a,exo_b) in zip(exo_coefficients,[(0,3),(3,4)]):
        exo_locale=((exo_p*exo_x[exo_a:exo_b]-exo_cibles[exo_a:exo_b])**2).mean()
        (exo_coefficient*exo_locale).backward()
    assert abs(exo_p.grad.item()+15) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [31]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()